# Analyzing Distortion in LMArena Setting

## Set-up

In [26]:
from datasets import load_dataset

ds = load_dataset("lmarena-ai/arena-human-preference-140k")

'TLS/SSL connection has been closed (EOF) (_ssl.c:1129)' thrown while requesting HEAD https://huggingface.co/datasets/lmarena-ai/arena-human-preference-140k/resolve/6322995ab34d7c2693e3f47dd13fa5caa0789a74/arena-human-preference-140k.py
Retrying in 1s [Retry 1/5].
Using the latest cached version of the dataset since lmarena-ai/arena-human-preference-140k couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /Users/jennifer/.cache/huggingface/datasets/lmarena-ai___arena-human-preference-140k/default/0.0.0/6322995ab34d7c2693e3f47dd13fa5caa0789a74 (last modified on Sat Mar  7 18:24:36 2026).


In [2]:
train_data = ds['train']

Import the utils that I wrote.

In [3]:
import utils_3 as ut

## Filtering Data via Vectors

In [4]:
ds_keys = [
    ('is_code',),
    ('category_tag', 'creative_writing_v0.1', 'creative_writing'),
    ('category_tag', 'criteria_v0.1', 'complexity'),
    ('category_tag', 'criteria_v0.1', 'creativity'),
    ('category_tag', 'criteria_v0.1', 'domain_knowledge'),
    ('category_tag', 'criteria_v0.1', 'problem_solving'),
    ('category_tag', 'criteria_v0.1', 'real_world'),
    ('category_tag', 'criteria_v0.1', 'specificity'),
    ('category_tag', 'criteria_v0.1', 'technical_accuracy'),

    ('category_tag', 'math_v0.1', 'math'),
    ('category_tag', 'if_v0.1', 'if'),
]

In [5]:
vector_len = len(ds_keys)

In [6]:
import numpy as np

def bit_strings(n):
    x = np.arange(2**n, dtype=np.uint32)[:, None]
    return ((x >> np.arange(n - 1, -1, -1)) & 1)

In [7]:
vector_keys = bit_strings(vector_len)
print(vector_keys)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 1]
 [0 0 0 ... 0 1 0]
 ...
 [1 1 1 ... 1 0 1]
 [1 1 1 ... 1 1 0]
 [1 1 1 ... 1 1 1]]


In [8]:
all_candidates = set(train_data['model_a']).union(set(train_data['model_b']))
all_candidate_idxs = {cand : i for i, cand in enumerate(all_candidates)}

In [24]:
import numpy as np
from tqdm import tqdm

def filter(ds_dict, keys, key_vector, language="en"):
    languages = np.asarray(ds_dict["language"])
    mask = languages == language

    for path, target in tqdm(list(zip(keys, key_vector)), total=len(keys)):
        values = ds_dict
        for k in path:
            values = values[k]
        mask &= np.asarray(values) == target

    return ds_dict[mask], mask

In [25]:
def process(ds_dict, all_candidate_idxs):

    w = ds_dict['winner']
    valid_mask = (w == 'model_a') | (w == 'model_b')

    a = ds_dict['model_a'][valid_mask]
    b = ds_dict['model_b'][valid_mask]

    a_idx = np.vectorize(all_candidate_idxs.get)(a)
    b_idx = np.vectorize(all_candidate_idxs.get)(b)

    winners = np.where(winners == 'model_a', a_idx, b_idx)
    losers = np.where(winners == 'model_a', b_idx, a_idx)

    return winners, losers

## Computing Distortion and Misspecification Metrics

In [ ]:
n_items = len(all_candidate_idxs)

In [ ]:
key_vector = vector_keys[5]

filtered_ds, _ = filter(train_data, ds_keys, key_vector)
winners, losers = process(filtered_ds, all_candidate_idxs)

r_hat, result = ut.fit_bradley_terry(winners, losers, n_items)


array([ True, False, False,  True])

In [16]:
a = np.array(
    [2, 2, 2]
)

b = np.array(
    [3, 3, 3]
)

c = np.array(
    [4, 4, 4]
)

In [22]:
np.stack((a, b, c)).T

array([[2, 3, 4],
       [2, 3, 4],
       [2, 3, 4]])

array([ True, False, False])